# Adım 3 — Veriyi hazırlayalım: birleştir, temizle, tekilleştir, kaydet

**Önceki adımdan ne öğrendik?** 6 veri setinin hepsi açıldı. Bunlardan **3'ünü ana veri** olarak seçtik, **2'sini dış test** olarak ayırdık, **1'ini kullanmıyoruz**:

| Rol | Veri seti | Neden |
|---|---|---|
| **Ana** | `xTRam1/safe-guard-prompt-injection` (~10 bin) | Büyük, temiz `text` + `label` (0/1) |
| **Ana** | `neuralchemy/Prompt-injection-dataset` (~6 bin) | Ek sütunları var: `category` (saldırı türü), `source`, `group_id` |
| **Ana** | `wambosec/prompt-injections` (~6 bin) | Ek sütunları var: `category`, `goal`; uzun saldırı örnekleri |
| **Dış test** | `jackhhao/jailbreak-classification` | Jailbreak tarzı (başka saldırı türü). **Asla eğitime girmez**, sadece test. Etiket `type` sütununda yazı olarak ("jailbreak"/"benign"). |
| **Dış test** | `deepset/prompt-injections` | Çok küçük (116 test) ve çok dilli (Almanca vb.). **Asla eğitime girmez.** Sonucu "ek bilgi" olarak raporlanır. |
| Kullanmıyoruz | `guychuk/benign-malicious-prompt-classification` | 464 bin satır, büyük ihtimalle başka veri setlerinden derlenmiş. Diğerleriyle çakışma riski yüksek ve çok büyük. |

**Bu defterin yaptığı:**
1. 5 veri setini aynı şemaya çevirir: `text, label, kaynak, rol, bolum, kategori, grup`. (`label`: 0 = zararsız, 1 = zararlı)
2. Boş metinleri atar.
3. Aynı veri setinde **çelişen etiketli** metinleri atar, **tekrarlayanlardan** birini tutar.
4. **Veri setleri arası ortak metinleri** bulur ve atar. Bunlar kalırsa "görmediği veri seti" testi sahte olurdu (model aynı cümleyi eğitimde görmüş olur). Özette söz verdiğimiz "veri sızıntısını önleme" bu.
5. `veri_birlesik.csv` dosyasını kaydeder.

**Çalıştırma:** Üst menü → `Çalışma zamanı → Tümünü çalıştır`. Sonunda dosya bilgisayarınıza iner. Ekran çıktısını Claude'a yapıştırın.

In [ ]:
!pip -q install datasets pandas

## 1) Hangi veri seti hangi sütunlarda ne tutuyor?
Bu tablo `01` defterinde gördüklerimizden çıktı. `etiket` sütunu bazen `label` (0/1), bazen `type` (yazı).

In [ ]:
import pandas as pd
from datasets import load_dataset

KAYNAKLAR = [
    dict(ad="xTRam1/safe-guard-prompt-injection",    kisa="safeguard",   rol="ana",      metin="text",   etiket="label", kategori=None,       grup=None),
    dict(ad="neuralchemy/Prompt-injection-dataset",  kisa="neuralchemy", rol="ana",      metin="text",   etiket="label", kategori="category", grup="group_id"),
    dict(ad="wambosec/prompt-injections",            kisa="wambosec",    rol="ana",      metin="prompt", etiket="label", kategori="category", grup=None),
    dict(ad="jackhhao/jailbreak-classification",     kisa="jackhhao",    rol="dis_test", metin="prompt", etiket="type",  kategori=None,       grup=None),
    dict(ad="deepset/prompt-injections",             kisa="deepset",     rol="dis_test", metin="text",   etiket="label", kategori=None,       grup=None),
]

def etiket_cevir(deger):
    # Her türlü etiketi 0 veya 1'e çevirir. Tanımadığı bir değer görürse HATA verir (sessizce yanlış yapmasın diye).
    s = str(deger).strip().lower()
    return {"0": 0, "1": 1, "benign": 0, "jailbreak": 1}[s]

## 2) Hepsini indirip aynı şemaya çevirelim

In [ ]:
parcalar = []
for k in KAYNAKLAR:
    ds = load_dataset(k["ad"])
    for bolum in ds.keys():                      # train / test / validation
        df = ds[bolum].to_pandas()
        yeni = pd.DataFrame({
            "text":     df[k["metin"]].astype(str),
            "label":    df[k["etiket"]].map(etiket_cevir),
            "kaynak":   k["kisa"],
            "rol":      k["rol"],
            "bolum":    "val" if bolum == "validation" else bolum,
            "kategori": df[k["kategori"]].astype(str) if k["kategori"] else "",
            "grup":     df[k["grup"]].astype(str) if k["grup"] else "",
        })
        parcalar.append(yeni)

ham = pd.concat(parcalar, ignore_index=True)
print("Birleştirilmiş ham satır sayısı:", len(ham))
ham.groupby(["rol", "kaynak", "bolum"]).size()

## 3) Temizlik
- Boşlukları kırp, boş metinleri at.
- Karşılaştırma için bir **anahtar** üretiyoruz: metnin küçük harfli, fazla boşlukları silinmiş hali. "Hello  World" ile "hello world" aynı cümle sayılsın.

In [ ]:
ham["text"] = ham["text"].str.strip()
ham = ham[ham["text"].str.len() > 0].copy()
ham["anahtar"] = ham["text"].str.lower().str.replace(r"\s+", " ", regex=True)
satir_ham = len(ham)
print("Boş metinler atıldıktan sonra:", satir_ham)

## 4) Aynı veri seti içinde tekrar ve çelişki
- Aynı metin hem 0 hem 1 etiketliyse **hangisi doğru bilemeyiz**, ikisini de atıyoruz.
- Aynı metin aynı etiketle birden çok kez geçiyorsa birini tutuyoruz.

In [ ]:
etiket_sayisi = ham.groupby(["kaynak", "anahtar"])["label"].nunique()
cakisanlar = etiket_sayisi[etiket_sayisi > 1].index
print("Aynı veri setinde çelişen etiketli metin sayısı:", len(cakisanlar))
ham = ham[~ham.set_index(["kaynak", "anahtar"]).index.isin(cakisanlar)]

onceki = len(ham)
ham = ham.drop_duplicates(subset=["kaynak", "anahtar"])
print("Veri seti içi tekrar silindi:", onceki - len(ham), "satır")

## 5) Veri setleri ARASI ortak metinler (veri sızıntısı kontrolü)
Bir metin iki farklı veri setinde varsa, birinde eğitip ötekinde test ettiğimizde model cevabı **ezberlemiş** olur. Bu yüzden ortak metinleri **her yerden** atıyoruz ve kaç tane olduğunu raporluyoruz.

In [ ]:
kac_kaynakta = ham.groupby("anahtar")["kaynak"].nunique()
ortak = set(kac_kaynakta[kac_kaynakta > 1].index)
print("Birden fazla veri setinde geçen metin sayısı:", len(ortak))

if ortak:
    hangi_ciftler = (ham[ham["anahtar"].isin(ortak)]
                     .groupby("anahtar")["kaynak"].apply(lambda s: " + ".join(sorted(s))))
    print("Hangi veri setleri arasında?")
    print(hangi_ciftler.value_counts().to_string())

onceki = len(ham)
ham = ham[~ham["anahtar"].isin(ortak)]
print("Ortak metinler atıldı:", onceki - len(ham), "satır")

## 6) neuralchemy için ek kontrol
Bu veri setinde `group_id` var: aynı cümlenin değiştirilmiş (augmented) kopyaları aynı gruba ait olabilir. Aynı grup hem eğitimde hem testte geçerse yine sızıntı olur. Kaç grup birden fazla bölümde görünüyor?

In [ ]:
n = ham[(ham["kaynak"] == "neuralchemy") & (ham["grup"] != "")]
bolum_sayisi = n.groupby("grup")["bolum"].nunique()
print("Birden fazla bölümde (train/val/test) görünen group_id sayısı:", int((bolum_sayisi > 1).sum()))
print("Toplam group_id sayısı:", n["grup"].nunique())

## 7) Sonuç: temiz veri
Bu tabloyu ve altındaki kategori listelerini Claude'a yapıştıracaksınız.

In [ ]:
ozet = (ham.groupby(["rol", "kaynak", "bolum"])
          .agg(satir=("label", "size"), zararli=("label", "sum"))
          .reset_index())
ozet["zararli_orani"] = (ozet["zararli"] / ozet["satir"]).round(2)
print("TEMİZ VERİ ÖZETİ")
print(ozet.to_string(index=False))
print()
print("Toplam satır: önce", satir_ham, "→ sonra", len(ham))
for ad in ["neuralchemy", "wambosec"]:
    print()
    print(f"{ad} kategorileri (kaç satır):")
    print(ham[ham["kaynak"] == ad]["kategori"].value_counts().to_string())

## 8) Kaydedelim
Dosya `veri_birlesik.csv` adıyla bilgisayarınıza iner. Sonra onu Drive'daki **YiitliProje** klasörüne sürükleyin, Ahmet de kullanacak.

In [ ]:
cikis = ham[["text", "label", "kaynak", "rol", "bolum", "kategori", "grup"]]
cikis.to_csv("veri_birlesik.csv", index=False)
print("Kaydedildi:", cikis.shape)

from google.colab import files
files.download("veri_birlesik.csv")